# **Vencimiento Automático de Solicitudes**
___
El Objetivo del Código es cerrar de forma automática las Solicitudes con un tiempo de Espera de 30 días hábiles.

<br>

Esto garantiza que no exista una saturación de solicitudes pero al costo de dejar de atenderlas.

## **Condiciones de Cerrar Solicitud**
___
Las Condiciones de Cierre Automático de Solicitud son:
- **15 días hábiles sin Respuesta** y
- **15 días hábiles después de Solicitado**
- **Todas las Deudas se encuentran Liquidadas**: Para este caso se actualiza el Comentario del Negociador en Caso de que alguna se encuentre como Liquidada

# **Imports Necesarios**
___
Los Códigos Necesarios para que el Aplicativo Funcione Correctamente

In [1]:
import pandas as pd
import gspread
from datetime import datetime
import json
from gspread_dataframe import get_as_dataframe, set_with_dataframe
from google.oauth2.service_account import Credentials
from gspread.exceptions import APIError
import numpy as np
from time import sleep
import os
import requests
from requests.exceptions import ChunkedEncodingError, ConnectionError, Timeout
from io import StringIO
# Importamos el diccionario con valores predefinidos
from googleapiclient.discovery import build
from collections import defaultdict
import re
import io
from typing import Literal
import holidays

# Agregamos el offset para que tenga en cuenta solo business days
co_holidays = holidays.country_holidays('CO', years=pd.Timestamp.now().year)
co_bday = pd.offsets.CustomBusinessDay(holidays=co_holidays) # type: ignore

# Función Auxiliar para reintentar cualquier llamda al API de sheets
def _retry(fn, label="", tries=10, base_sleep=1.5, jitter=0.6, max_sleep=45):
    RETRIABLE_CODES = ["[500]", "[502]", "[503]", "[504]", "[429]"]
    last_err = None
    for i in range(tries):
        try:
            return fn()
        except APIError as e:
            last_err = e
            msg = str(e)
            # Si el error fue del servidor
            if any(c in msg for c in RETRIABLE_CODES):
                sleep_s = min((base_sleep) * (2 ** i) + np.random.uniform(0, jitter), max_sleep)
                print(f"[RETRY {i+1}/{tries}] {label} -> {msg[:120]}... sleep {sleep_s:.1f}s")
                # Esperamos para no saturar al API
                sleep(sleep_s)
                continue
            raise
    raise last_err

# Funcion Auxiliar para obtener variables de entorno/ secretos de forma robusta
def getEnvVar(var: str):
  try: # Si el entorno es Colab las credenciales se obtendrán de esa forma
    from google.colab import userdata
    return userdata.get(var)
  except ImportError: # Si el entorno es GitHub las credenciales se obtienen es mediante variables de entorno
    var = os.getenv(var)
    if not var:
      raise ValueError("No se encontraron las credenciales en las variables de entorno.")
    return var

def getColLetter(n: int) -> str:
  """
  Convert a 1-based column number to an Excel column letter.
  """
  result = []
  while n > 0:
    n, remainder = divmod(n - 1, 26)
    result.append(chr(65 + remainder))
  return "".join(reversed(result))

# Función Auxiliar para transformar todas las columnas númericas a float
def numericToFloat(df: pd.DataFrame):
  # Filtramos las columnas númericas
  numericCols = df.select_dtypes(include=[np.number, 'bool']).columns
  # Creamos una copia para que no salga el warning de pandas
  nuevoDF = df.copy()
  # Convertimos todas esas columnas en float
  nuevoDF[numericCols] = nuevoDF[numericCols].astype(float)
  return nuevoDF

# Función Auxiliar para obtener credenciales
def getCreds():
  try: # Si el entorno es Colab las credenciales se obtendrán de esa forma
    from google.colab import userdata
    return json.loads(userdata.get('MI_JSON'))
  except ImportError: # Si el entorno es GitHub las credenciales se obtienen es mediante variables de entorno
    creds = os.getenv('MI_JSON')
    if not creds:
      raise ValueError("No se encontraron las credenciales en las variables de entorno.")
    return json.loads(creds.strip())

# Función Auxiliar para imputar NaNs
def imputeNans(df: pd.DataFrame, col: str, value):
  # Se define la mascara de valores nulos
  mask = df[col].isna()
  # A los valores nulos se aplica el valor
  df.loc[mask, col] = value

# Función Auxiliar para obtener un DF a partir de la hoja y el rango
def gettingAsDF(ws, cellRange: str) -> pd.DataFrame:
  # Obtenemos los valores del rango propuesto
  values = _retry(lambda: ws.get(cellRange, pad_values=True))
  # Definimos los headers y las filas
  headers = values[0]
  rows = values[1:]
  # Creamos el DF
  df = pd.DataFrame(rows, columns=headers)
  return df

# Función Auxiliar para normalizar columnas
def normalizeCols(df: pd.DataFrame) -> pd.DataFrame:
  # Las transformaciones que se van a realizar serán: Quitar tildes, Quitar espacios y dejar todo con .title
  originalCols = df.columns.tolist()
  newCols = []
  for col in originalCols:
    # Primero quitamos todas las tildes
    newColName = col.lower().replace("ó", "o").replace("á", "a").replace("í", "i").replace("é", "e").replace("ú", "u")
    # Ahora lo dejamos como titulo y quitamos espacios sobrantes
    newColName = newColName.strip().title()
    newCols.append(newColName)

  # Cambiamos las columnas en el DF
  df.columns = newCols
  return df

# Función Auxiliar para Limpiar Números
def cleanNumber(value, default_nan: float = 0.0) -> float:
    if not isinstance(value, str):
        return value
    # Reemplazamos X por ''
    value = value.replace('X','')

    # Check if the string contains numbers and currency symbols/commas
    # This regex matches things like: "$ 1.234,56", "50,000", or "1.200.000,00"
    clean_val = value.replace('$', '').replace(' ', '')

    try:
        # Common in Latin America: 1,000.00 -> 1000.00
        if (',' in clean_val) and ('.' in clean_val) and (clean_val.index(',') > clean_val.index('.')):
            clean_val = clean_val.replace('.', '').replace(',', '.')
        elif (',' in clean_val) and ('.' in clean_val) and (clean_val.index(',') < clean_val.index('.')):
            # Reemplzamos , con nada
            clean_val = clean_val.replace(',', '')
        elif '.' in clean_val and clean_val.count('.') > 1:
            # Handle "666.666.666" as 666666666
            clean_val = clean_val.replace('.','')
        elif ',' in clean_val and clean_val.count(',') > 1:
            # Handle "666.666.666" as 666666666
            clean_val = clean_val.replace(',','')
        elif ',' in clean_val and clean_val.count(',') == 1:
            parts = clean_val.split(',')
            # The first part can be 1-3 digits; all following parts MUST be exactly 3 digits
            if 1 <= len(parts[0]) <= 3 and all(len(s) == 3 for s in parts[1:]):
                clean_val = clean_val.replace(',', '')
            else:
                # If it's a decimal separator used multiple times (e.g., European format typo or specific notation)
                # Note: A valid float can only have ONE decimal point.
                # If there are multiple commas acting as decimals, replacing them all with '.' will still error out.
                clean_val = clean_val.replace(',', '.')
        elif '.' in clean_val and clean_val.count('.') == 1:
            # 2 Possible Cases: Decimal Separator or Thousand Separator
            # Detecting case thousand by: splitting and Confirming len of all == 3
            if all([len(s) == 3 for s in clean_val.split('.')]):
                clean_val = clean_val.replace('.', '')
            # Else it's decimal so its ignored
        return float(clean_val)
    except ValueError:
        clean_val = pd.to_numeric(clean_val, errors='coerce')
        return clean_val if pd.notna(clean_val) else default_nan  # Not a number? Return original text

# Función Auxiliar para Obtener la Diferencia en Días Hábiles en float entre dos fechas
def getBDDaysDiffFloat(firstDate: pd.Timestamp, secondDate: pd.Timestamp, change_order: bool = True) -> float:
    # Verificamos que no sean NaT
    if pd.isna(firstDate) or pd.isna(secondDate):
        return np.nan

    # Quitamos Zonas Horarias
    firstDate = firstDate.tz_localize(None) if firstDate.tzinfo else firstDate
    secondDate = secondDate.tz_localize(None) if secondDate.tzinfo else secondDate

    # Verificamos el Orden Correcto de Fechas
    if change_order:
        start, end = sorted([firstDate, secondDate])
    else:
        start, end = firstDate, secondDate

    # Verificamos que la Comparación sea Correcta
    if start > end:
        return 0

    # 1. Normalize dates to calculate 'full' business days in between
    # We use 'floor' to get the count of midnight-to-midnight periods
    full_bus_days = len(pd.date_range(start=start.floor('D'), end=end.floor('D'), freq=co_bday)) - 1

    # 2. Handle the fractional part of the start and end days
    # We subtract the time elapsed on the first day and add time elapsed on the last
    # This assumes a "day" is 24 hours.
    start_fraction = (start.hour + start.minute / 60 + start.second / 3600) / 24
    end_fraction = (end.hour + end.minute / 60 + end.second / 3600) / 24

    # Total = Full Days - (Time passed on start day) + (Time passed on end day)
    float_diff = full_bus_days - start_fraction + end_fraction

    return round(float_diff, 4)

def clean_inserted_at(df, column_name="inserted_at"):
    """
    Standardizes dates while fixing the 'Columns must be same length as key' error.
    """
    # Create a copy to avoid modifying the original dataframe in place
    df = df.copy()

    # 1. Clean the strings (remove whitespace, handle NaNs)
    s_str = df[column_name].astype(str).str.strip()

    # 2. Initialize the result series with NaT
    # We use the index from the original dataframe to ensure alignment
    result = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")

    # 3. Process Hyphens: YYYY-MM-DD (ISO/Standard)
    mask_hyphen = s_str.str.contains("-", na=False)
    if mask_hyphen.any():
        # result.loc[mask_hyphen] needs data that matches the mask_hyphen length
        result.loc[mask_hyphen] = pd.to_datetime(
            s_str.loc[mask_hyphen],
            errors="coerce"
        )

    # 4. Process Slashes: DD/MM/YYYY
    mask_slash = s_str.str.contains("/", na=False)
    if mask_slash.any():
        # Using .loc[mask_slash] on both sides ensures the indices align perfectly
        result.loc[mask_slash] = pd.to_datetime(
            s_str.loc[mask_slash],
            dayfirst=True,
            errors="coerce"
        )

    # 5. Assign back to the dataframe
    df[column_name] = result

    return df

# Función Auxiliar para Permitir diferentes Nombres de Columnas
def cleanCols(df: pd.DataFrame, realCol: str, colPossibleVals: list = []) -> pd.DataFrame:
  dfCols = df.columns.tolist()
  # Por cada una de las Columnas Posibles
  for col in colPossibleVals:
    # Si existe se actualiza y se termina la Ejecución
    if col in dfCols:
      if col != realCol and realCol in dfCols:
        df = df.drop(columns=[realCol])
      df = df.rename(columns={col: realCol})
      break
  else:
    print('✖️No se realizó el cambió para {} con {} posibles valores.'.format(realCol, ', '.join(colPossibleVals)))
    df[realCol] = np.nan
  return df

def print_grid(strings, n, sep):
    """
    Prints a list of strings with 'n' items per row,
    separated by 'sep'.
    """
    for i, word in enumerate(strings):
        # Print the word. Use end='' to manage the separator manually.
        print(word, end="")

        # Check if we need a separator or a newline
        if (i + 1) % n == 0:
            print()  # Newline after every n-th element
        elif i < len(strings) - 1:
            print(sep, end="") # Only print separator if not at the end of a row

    # Add a final newline if the last row wasn't "full"
    if len(strings) % n != 0:
        print()

# Función Auxiliar para limpiar valores
def cleanText(txt):
  return txt.lower().replace("ó", "o").replace("á", "a").replace("í", "i").replace("é", "e").replace("ú", "u").upper().strip()

# Función Auxiliar para Limpiar Nombres de Casas de Cobro
def cleanCasa(txt):
  if txt == 'JERFERSON CAPITAL' or txt == 'JEFERSON CAPITAL':
    return 'JCAP'
  elif txt == 'AVVILLAS' or 'AV VILLAS' in txt:
    return 'AV VILLAS'
  elif txt == 'CESS BBVA':
    return 'CESS'
  elif txt == 'FREE MANAGMENT':
    return 'FREE MANAGEMENT'
  elif txt == 'INVERCIONISTAS':
    return 'INVERSIONISTAS'
  elif txt == 'REINCAR AV VILLAS':
    return 'REINCAR'
  elif 'SISTEMGROUP' in txt:
    return 'SYSTEMGROUP'
  return txt

# Funcion Auxiliar para Escribir Datos en una celda específica
def writeToCell(ws, row, col, value):
  _retry(lambda: ws.update_cell(row, col, value), label=f"Update Cell ({row},{col})")

def getBDDaysDiffFloat(firstDate: pd.Timestamp, secondDate: pd.Timestamp) -> float:
  # Verificamos que no sean NaT
  if pd.isna(firstDate) or pd.isna(secondDate):
    return np.nan
  # Ensure dates are in the correct order
  start, end = sorted([firstDate, secondDate])

  # 1. Normalize dates to calculate 'full' business days in between
  # We use 'floor' to get the count of midnight-to-midnight periods
  full_bus_days = len(pd.date_range(start=start.floor('D'), end=end.floor('D'), freq=co_bday)) - 1

  # 2. Handle the fractional part of the start and end days
  # We subtract the time elapsed on the first day and add time elapsed on the last
  # This assumes a "day" is 24 hours.
  start_fraction = (start.hour + start.minute / 60 + start.second / 3600) / 24
  end_fraction = (end.hour + end.minute / 60 + end.second / 3600) / 24

  # Total = Full Days - (Time passed on start day) + (Time passed on end day)
  float_diff = full_bus_days - start_fraction + end_fraction

  return round(float_diff, 4)

# Función Auxiliar para Listar los Archivos presentes en una carpeta de Drive
def listFiles(driveService, folderId):
  results = driveService.files().list(
        q=f"'{folderId}' in parents and trashed = false",
        fields="files(id, name, mimeType)"
    ).execute()
  files = results.get('files', [])
  return files

mesesDict = {
    1: 'Enero',2: 'Febrero',3: 'Marzo',4: 'Abril',5: 'Mayo',6: 'Junio',
    7: 'Julio',8: 'Agosto',9: 'Septiembre',10: 'Octubre',11: 'Noviembre',12: 'Diciembre'
}

# 1. Retrieve the secret from Colab
service_account_info = getCreds()

# 2. Define the scope and authenticate
scope = ['https://www.googleapis.com/auth/spreadsheets',
         'https://www.googleapis.com/auth/drive']

creds = Credentials.from_service_account_info(service_account_info, scopes=scope)
client = gspread.authorize(creds)

# Creamos el Servicio de Google Drive
drive_service = build('drive', 'v3', credentials=creds)

today = pd.Timestamp.now('America/Bogota').tz_localize(None)

# **Cambios de Referencia**
___
Los datos que se obtienen son: **refChangesDict**

In [2]:
# Abrimos la SpreadSheet
refChangesShId = '1jcPPhtF2YK3Kr7P_A0Mgh2OqhOfnVWB2to3UPoSH5tE'
refChangesSH = _retry(lambda: client.open_by_key(refChangesShId))
# Aca se cargan los cambios de referencia
refChangesWS = _retry(lambda: refChangesSH.worksheet('Cambios de Referencia'))
records = _retry(lambda: refChangesWS.get_all_values())

## La llave sera la referencia vieja y el valor la referencia nueva
if len(records)>0 and len(records[0])>1:
  refChangesDict = {str(row[0]).replace('.0','').strip():str(row[1]).replace('.0','').strip() for row in records}
else:
  refChangesDict = {}

print('✅Cambios de Referencia Cargados con Éxito')
print('ℹ️Cargas Totales: {}'.format(
    len(refChangesDict)
))

✅Cambios de Referencia Cargados con Éxito
ℹ️Cargas Totales: 3712


# **Obtención de Datos**
___
Se Obtienen los Datos de las Solicitudes del Formulario de Alianzas y de Liquidaciones MEC.

## **Datos del Formulario de Alianzas**
___
Los datos que se obtienen son: **solsDF**


In [3]:
# Paso 1: Abrir la Spreadsheet
solsSHId = '1tlHeLPJgIlRw3-_yv8lG4_w07n44o6KUxwxS1jmhjLk'
solsSH = _retry(lambda: client.open_by_key(solsSHId))
# Paso 2: Abrir la Worksheet
solsWS = _retry(lambda: solsSH.worksheet('Solicitudes_MEC'))
# Paso 3: Obtener los Datos como un DF
solsDF = _retry(lambda: get_as_dataframe(solsWS, evaluate_formulas=True))

solsHeaders = solsDF.columns.tolist()

# Agregamos Row_Sheets como .index+2
solsDF['Row_Sheets'] = solsDF.index + 2

# Paso 4: Limpieza de Datos
# Volvemos las Columnas Necesarias a Timestamp
for col in ['Timestamp', 'Fecha_Esperada_Pago', 'Fecha_Respuesta', 'Fecha_Limite_Pago']:
  solsDF[col] = pd.to_datetime(solsDF[col], errors='coerce', dayfirst=False)

# Volvemos las Columnas Referencia, ID_Solicitud y Cedula a String
for col in ['Referencia', 'ID_Solicitud', 'Cedula']:
  solsDF[col] = solsDF[col].apply(lambda s: str(s).replace('.0','').strip() if pd.notna(s) else '')

# Aplicamos los Cambios de Referencia
solsDF['Referencia'] = solsDF['Referencia'].apply(lambda s: refChangesDict.get(s,s))

# Imputamos Ejecutivo con 'Sin Asignar'
imputeNans(solsDF, 'Ejecutivo', 'Sin Asignar')

# Hacemos Parsing de la Columna Datos_Solicitud y Metadata_Solicitud a JSON
for col in ['Datos_Solicitud', 'Metadata_Solicitud','JSON_Respuesta']:
  solsDF[col] = solsDF[col].apply(lambda s: json.loads(s) if pd.notna(s) else {})

# Mostramos el Resultado
print('✅Datos de Solicitudes traídos con éxito, Columnas: {}'.format(
    ', '.join(solsDF.columns)
))
print('ℹ️Filas Totales: {}'.format(len(solsDF)))

✅Datos de Solicitudes traídos con éxito, Columnas: ID_Solicitud, Timestamp, Correo, Referencia, Cedula, Ids_Deuda, Casa_Cobro, Tipo_Solicitud, Datos_Solicitud, Fecha_Esperada_Pago, Tipo_Pago, Ejecutivo, Metadata_Solicitud, Estado_Solicitud, Fecha_Respuesta, Fecha_Limite_Pago, JSON_Respuesta, Row_Sheets
ℹ️Filas Totales: 683


## **Datos de Liquidaciones del Mes**
___
Datos que se obtienen: **deudasLiq: set[str]**

In [4]:
# Abrimos la SpreadSheet
liqsSHId = '1H3sYEtkeu47POnu8xZMaMtID1Vj53YIcWblWeZ8d0rc'
liqsSH = _retry(lambda: client.open_by_key(liqsSHId))

# Se define el rango de columnas a obtener
# PARAMETRO CAMBIABLE -----------------
cellRange = 'A1:U'

minYear = today.year - 1

# Se definen las columnas a usar de las a modo de Diccionario
# PARAMETRO CAMBIABLE -----------------
colsliqs = {
    'Id_Deuda': ['Deuda Berex'],
}

# ------------ liqs Mensuales --------------
nombreHojaliqs = 'BD del mes'
liqs = _retry(lambda: liqsSH.worksheet(nombreHojaliqs))
liquidacioneMensualessDF = gettingAsDF(liqs, cellRange)

# --- Limpieza de Columnas ---
for col, possibleVals in colsliqs.items():
  liquidacioneMensualessDF = cleanCols(liquidacioneMensualessDF, col, possibleVals)

# Dejamos Solo las Columnas Necesarias
liquidacioneMensualessDF = liquidacioneMensualessDF[list(colsliqs.keys())]
# Creamos la Columna Or_Liq = 'Mes'
liquidacioneMensualessDF['Or_Liq'] = 'Mes'

print('🆔Liquidaciones del Mes Cargadas con Éxito, {} filas'.format(len(liquidacioneMensualessDF)))

# --------- liqs Anuales -------------------
nombresHojasliqs = ['BD ' + str(y) for y in range(minYear, today.year+1)]

liqDFList = []

for name in nombresHojasliqs:
  liqs = _retry(lambda: liqsSH.worksheet(name))
  liqsDF = gettingAsDF(liqs, cellRange)
  # --- Limpieza de Columnas ---
  for col, possibleVals in colsliqs.items():
    liqsDF = cleanCols(liqsDF, col, possibleVals)
  # Dejamos las Columnas Necesarias
  liqsDF = liqsDF[list(colsliqs.keys())]
  # Creamos la Columna Or_Liq = 'Year'
  liqsDF['Or_Liq'] = 'Year'
  # Agregamos el DF a la Lista
  liqDFList.append(liqsDF)
  print('🆔Liquidaciones para {} cargadas con éxito: {} filas'.format(name, len(liqsDF)))

liqsAnualesDF = pd.concat(liqDFList, ignore_index=True)

# Combinar liqs de Año con liqs Mensuales
liqsDF = pd.concat([liqsAnualesDF, liquidacioneMensualessDF], ignore_index=True)

# Quitamos Datos donde Id_Deuda sea NaN
liqsDF = liqsDF.dropna(subset=['Id_Deuda'])

# Volvemos Id_Deuda y Ref_Liq a String
liqsDF['Id_Deuda'] = liqsDF['Id_Deuda'].apply(lambda s: str(s).replace('.0',''))

# Volvemos las Deudas a un Set
deudasLiq = set(liqsDF['Id_Deuda'].tolist())

# Imprimimos para logs
print('✅Liquidaciones Cargadas con éxito')
print('🚼Deudas totales: {} filas.'.format(len(deudasLiq)))

🆔Liquidaciones del Mes Cargadas con Éxito, 37 filas
🆔Liquidaciones para BD 2025 cargadas con éxito: 17809 filas
🆔Liquidaciones para BD 2026 cargadas con éxito: 9854 filas
✅Liquidaciones Cargadas con éxito
🚼Deudas totales: 25739 filas.


# **Filtrado de Datos**
___
Se identifican las Solicitudes a Cerrar.

## **Funciones Auxiliares de Ayuda de Filtrado**
___
Funciones para facilitar la lectura de la lógica del código.

In [5]:
def es_solicitud_sin_responder(solicitud: pd.Series) -> bool:
    """
    Determina si una solicitud específica no ha sido respondida.

    Args:
        solicitud (pd.Series): Información de la solicitud.

    Returns:
        bool: True si la solicitud no ha sido respondida, False en caso contrario.
    """
    maskSinTocar = (solicitud["Estado_Solicitud"] == "Sin Tocar") | (solicitud["Estado_Solicitud"] == "Solicitado")
    maskBajoComite = solicitud["Metadata_Solicitud"].get("Estado_Comite", 0) == 3 and solicitud["Estado_Solicitud"] == "Bajo Comité"
    maskTitularIlocalizable = solicitud["Metadata_Solicitud"].get("Estado_Titular_Ilocalizable", 0) == 3 and solicitud["Estado_Solicitud"] == "Titular Ilocalizable"
    return (maskSinTocar or maskBajoComite or maskTitularIlocalizable)

# Máscara 1: Fecha_Solicitado en Metadata mayor a 15 días hábiles
def solicitado_more_15_days(row: pd.Series) -> int:
  # 0: No tiene fecha solicitado
  # 1: Tienen fecha solicitado pero no cumple
  # 2: Tiene fecha solicitado y si cumple

  # Primera Verificación: Que no este respondida (Sin Tocar, Solicitada o Aprobacion Necesaria)
  if not es_solicitud_sin_responder(row):
    return 0
  # Segunda Verificación: La Metadata_Solicitud tiene Fecha_Solicitado
  if row["Metadata_Solicitud"].get("Fecha_Solicitado", None) is None:
    return 0

  # Ahora volvemos la Fecha a Datetime con dayfirst = False
  fecha_solicitado = pd.to_datetime(row["Metadata_Solicitud"]["Fecha_Solicitado"], dayfirst=False)
  # Calculamos la Diferencia en Días Hábiles (float) a hoy
  diff_days = getBDDaysDiffFloat(fecha_solicitado, today)

  # Tercera Verificación: Cumplimiento de los 15 días
  if diff_days >= 15:
    return 2
  else:
    return 1

# Máscara 2: Solicitado hace más de 30 días
def subido_more_15_days(row: pd.Series) -> bool:
  # Obtenemos la diferencia entre row['Timestamp'] y hoy
  diff_days = getBDDaysDiffFloat(row['Timestamp'], today)
  return (diff_days >= 15) & (es_solicitud_sin_responder(row))

# Máscara 3: Liquidaciones sin Responder
def liquidacion_sin_responder(row: pd.Series) -> int:
  # Verificamos que la Solicitud no se haya respondido aún
  if not es_solicitud_sin_responder(row):
    return 0
  # Primero Obtenemos los Ids de Datos_Solicitud
  ids_Sol = [d['Id_Deuda'] for d in row['Datos_Solicitud']]
  # Ahora miramos cuantos de estos estan liquidados
  liqCount = sum((id_s in deudasLiq for id_s in ids_Sol))
  # Verificación Última: Liquidaciones Totales, Parciales y Sin Liquidaciones
  if liqCount == len(ids_Sol):
    return 2
  elif liqCount > 0:
    return 1
  else:
    return 0

## **Ejecución del Filtrado de Resultados**
___
Se realiza la lógica de selección bajo las 2 máscaras:
- **1: Solicitado 15 días**
- **2: Subido 30 días**

In [33]:
# Creamos la Serie de Solicitado 15 dias
solsDF["Solicitado_15_Dias"] = solsDF.apply(solicitado_more_15_days, axis=1)
# Creamos la Serie de Subido 30 días
solsDF["Subido_15_Dias"] = solsDF.apply(subido_more_15_days, axis=1)
# Creamos la Serie de Estado_Liq
solsDF["Estado_Liq"] = solsDF.apply(liquidacion_sin_responder, axis=1)

# Siguiente: Definir las Máscaras de Cambio
# Máscara 1: Solicitado_15_Dias == 2
# Máscara 2: Solicitado_15_Dias == 0 & Subido_15_Dias == True
# Máscara 3: Estado_Liq == 2
# Máscara 4: Estado_Liq == 1
maskSol = solsDF["Solicitado_15_Dias"] == 2
maskSub = (solsDF["Solicitado_15_Dias"] == 0) & solsDF["Subido_15_Dias"]
maskLiqTotal = (solsDF["Estado_Liq"] == 2)
maskLiqParcial = (solsDF["Estado_Liq"] == 1)

print('😁Distribución de Cambios:')
print('  -ℹ️ Solicitado 15 días: {}'.format(maskSol.sum()))
print('  -🚹 Subido 15 días: {}'.format(maskSub.sum()))
print('  -💰 Liquidaciones Totales: {}'.format(maskLiqTotal.sum()))
print('  -💰 Liquidaciones Parciales: {}'.format(maskLiqParcial.sum()))

# Agregamos el Motivo
solsDF['Motivo_Cambio'] = pd.Series([[] for _ in range(len(solsDF))])
solsDF.loc[maskSol, 'Motivo_Cambio'].apply(lambda l: l.append('Solicitado hace más de 15 días Hábiles'))
solsDF.loc[maskSub, 'Motivo_Cambio'].apply(lambda l: l.append('Subido hace más de 15 días Hábiles'))
solsDF.loc[maskLiqTotal, 'Motivo_Cambio'].apply(lambda l: l.append('Liquidacion de Todas las Deudas'))
solsDF.loc[maskLiqParcial, 'Motivo_Cambio'].apply(lambda l: l.append('Liquidaciones Parciales'))

# Ahora Guardamos el DF que tenga alguno de esos dos cambios
cambiarDF = solsDF[(maskSol | maskSub | maskLiqTotal | maskLiqParcial)].copy()

print('🟢Datos Totales a Cambiar: {}'.format(
    len(cambiarDF)
))

😁Distribución de Cambios:
  -ℹ️ Solicitado 15 días: 0
  -🚹 Subido 15 días: 1
  -💰 Liquidaciones Totales: 2
  -💰 Liquidaciones Parciales: 1
🟢Datos Totales a Cambiar: 4


# **Cambio de Datos**
___
Se actualizan 3 cosas:
- **Fecha_Respuesta**: Se cambia por Hoy
- **Estado_Solicitud**: Se cambia a Vencida
- **Metadata-Comentario_Ejecutivo**: *"Solicitud cerrada de forma automática por vencimiento de respuesta (15 días hábiles)"*

In [34]:
def aplicarCambioFila(row: pd.Series) -> pd.Series:
  # Paso 1: Identificar los Motivos
  motivos = row['Motivo_Cambio']
  # Paso 2: Aplicar el Cambio de Liquidaciones Parciales (EXCLUSIVO)
  if ('Liquidaciones Parciales' in motivos) and (len(motivos) == 1):
    # Definimos los Ids que ya se liquidaron
    idsLiquidados = [d['Id_Deuda'] for d in row['Datos_Solicitud'] if (d['Id_Deuda'] in deudasLiq)]
    # Actualizamos el Comentario del Negociador para mostrar los Ids Liquidados
    row['Metadata_Solicitud']['Comentario_Negociador'] += "\n\nALERTA: Estos **IDs están Liquidados: {}**".format(
        ' | '.format(idsLiquidados)
    ).strip()
    return row

  # Paso 3: Actualizar lo Necesario
  row['Fecha_Respuesta'] = today
  row['Estado_Solicitud'] = 'Vencida' if not ('Liquidaciones Totales' in motivos) else 'Validada por Fuera'
  # Paso 4: Actualizar el Comentario del Negocioador
  newCommentary = "*Solicitud cerrada de forma automática por vencimiento*: **{}**".format(
      ' | '.join(motivos)
  ).strip()
  row['Metadata_Solicitud']['Comentario_Ejecutivo'] = newCommentary
  # Paso 5: Retornar la Fila Actualizada
  return row

if len(cambiarDF) > 0:
  # Aplicamos el Cambio a cada Fila
  cambiarDF = cambiarDF.apply(aplicarCambioFila, axis=1)

print('✅Cambios Aplicados con Éxito')

✅Cambios Aplicados con Éxito


# **Subida de Datos**
___
Se implementa la actualización de estas solicitudes en el archivo de Google Sheets.

## **Funciones Auxiliares de Subida de Datos**
___
Las Funciones que permiten una comunicación robusta con Google Sheets.

In [20]:
# Funcion auxiliar para unir filas actualizadas en una sola y poder realizar cambios enteros por chunks
def _make_consecutive_blocks(rownums_sorted: list[int], values_by_rownum: dict[int, list[str]]):
    """
    Agrupa filas consecutivas para reducir llamadas a la API.
    Retorna [(start_row, end_row, matrix_values)]
    """
    blocks = []
    if not rownums_sorted:
        return blocks

    start = prev = rownums_sorted[0]
    mat = [values_by_rownum[start]]

    for r in rownums_sorted[1:]:
        # Si la fila es adyacente a la anterior se uno como un bloque
        if r == prev + 1:
            mat.append(values_by_rownum[r])
            prev = r
        else:
        # Si no, entonces se guarda el bloque y se crea uno nuevo
            blocks.append((start, prev, mat))
            start = prev = r
            mat = [values_by_rownum[r]]
    # Se guarda el último bloque en memoria
    blocks.append((start, prev, mat))
    return blocks

def letter_to_col(col_str: str) -> int:
    """Convierte una letra de columna de Sheets (ej. 'A', 'Z', 'AA') a su número de índice 1-based."""
    num = 0
    for char in col_str.upper():
        num = num * 26 + (ord(char) - ord('A') + 1)
    return num


def col_to_letter(col_idx: int) -> str:
    """Convierte un índice numérico 1-based de columna a su letra correspondiente en Sheets."""
    result = ""
    while col_idx > 0:
        col_idx, remainder = divmod(col_idx - 1, 26)
        result = chr(65 + remainder) + result
    return result

def _batch_update_rows(ws, start_col_letter: str, end_col_letter: str, row_blocks: list[tuple[int,int,list[list[str]]]], cell_threshold: int = 10000):
    """
    Updates Google Sheets using batch_update to minimize API calls.
    Groups row_blocks into 'mega-batches' based on a cell_threshold.
    """

    current_batch_data = []
    current_cell_count = 0

    for (r1, r2, mat) in row_blocks:
        # Calculate cells in this specific block
        block_cells = len(mat) * len(mat[0]) if mat else 0
        rng = f"{start_col_letter}{r1}:{end_col_letter}{r2}"

        # Prepare the update object for this block
        update_item = {
            'range': rng,
            'values': mat
        }

        # Check if adding this block exceeds our threshold
        if current_cell_count + block_cells > cell_threshold and current_batch_data:
            # Execute the accumulated batch before starting a new one
            _execute_batch_retry(ws, current_batch_data)
            current_batch_data = []
            current_cell_count = 0
            sleep(0.5) # Slight breather between mega-batches

        current_batch_data.append(update_item)
        current_cell_count += block_cells

    # Final execution for any remaining data
    if current_batch_data:
        _execute_batch_retry(ws, current_batch_data)

def _execute_batch_retry(ws, data_list):
    """
    Helper to wrap the batch_update in your retry logic.
    """
    _retry(
        lambda: ws.batch_update(data_list, value_input_option="USER_ENTERED"),
        label=f"batch_update for {len(data_list)} ranges"
    )

def update_sheet_data_batch(
    ws,
    data: list[list],
    start_col_letter: str = "A",
    cell_threshold: int = 10000
) -> bool:
    """
    Actualiza Google Sheets de forma masiva y eficiente agrupando filas consecutivas
    y dividiendo los envíos en bloques según un umbral de celdas.

    :param ws: Objeto Worksheet de gspread.
    :param data: Lista de listas con estructura [Número_Fila_Sheets, col1, col2, ...].
    :param start_col_letter: Letra de la columna donde inicia el bloque de datos (por defecto 'A').
    :param cell_threshold: Umbral máximo de celdas por solicitud batch_update.
    :return: True si la actualización se completó con éxito, False de lo contrario.
    """
    if not data:
        print("❌ No hay datos para actualizar.")
        return True

    try:
        # 1. Extraer los números de fila y mapear sus respectivos valores
        values_by_rownum = {}
        num_cols = None

        for item in data:
            if not item:
                continue

            row_num = item[0]
            row_values = item[1:]

            # Registrar la cantidad de columnas basada en la primera fila no vacía
            if num_cols is None:
                num_cols = len(row_values)

            values_by_rownum[row_num] = row_values

        if not values_by_rownum or num_cols == 0:
            return True

        # 2. Ordenar las filas para poder detectar la consecutividad
        rownums_sorted = sorted(values_by_rownum.keys())

        # 3. Calcular la letra de la columna final según la longitud de los headers/datos
        start_col_idx = letter_to_col(start_col_letter)
        end_col_idx = start_col_idx + num_cols - 1 # type: ignore
        end_col_letter = col_to_letter(end_col_idx)

        # 4. Crear bloques de filas consecutivas
        row_blocks = _make_consecutive_blocks(rownums_sorted, values_by_rownum)

        # 5. Ejecutar la actualización masiva utilizando los bloques generados
        _batch_update_rows(
            ws=ws,
            start_col_letter=start_col_letter.upper(),
            end_col_letter=end_col_letter,
            row_blocks=row_blocks,
            cell_threshold=cell_threshold
        )

        return True

    except Exception as e:
        print(f"[Error] No se pudo completar la actualización en lote: {e}")
        return False

# Función Auxiliar para Convertir los Datos a String
def convert_data_to_string(obj) -> str:
    """
    Converts various data types to a string representation.
    Handles None, NaN, and other types gracefully.
    """
    if obj is None:
        return ""
    if isinstance(obj, str):
        return obj
    if isinstance(obj, pd.Timestamp) and pd.isna(obj):
        return ''
    if isinstance(obj, pd.Timestamp):
        return obj.strftime("%Y-%m-%d %H:%M:%S").replace("NaT","")
    if isinstance(obj, float) and np.isnan(obj):
        return ""
    if isinstance(obj, (int, float)):
        return str(obj)
    # For other types (like lists, dicts), we can use json.dumps for a readable format
    try:
        return json.dumps(obj, ensure_ascii=False)
    except TypeError:
        return str(obj).replace('\'','"')

## **Subida de Datos a Solicitudes_MEC**
___
Se aplica la actualización de los Datos

In [35]:
# Paso 1: Convertir el DF de Cambios a una lista de datos
finalHeaders = ['Row_Sheets'] + solsHeaders
cambiarDF = cambiarDF[finalHeaders]
cambiarMatrix = cambiarDF.values.tolist()

# Paso 2: Convertir los Datos a "Sheets accepted"
final_changes = []
for row in cambiarMatrix:
  if row:
    row_cleaned = [row[0]] + [convert_data_to_string(cell) for cell in row[1:]]
    final_changes.append(row_cleaned)

# Aplicamos los Cambios
update_sheet_data_batch(
    ws=solsWS,
    data=final_changes,
    start_col_letter='A',
    cell_threshold=10000
)

print('✅Cambios Aplicados con Éxito')
print('ℹ️Cambios Totales: {}'.format(
    len(final_changes)
))

✅Cambios Aplicados con Éxito
ℹ️Cambios Totales: 4
